# บทที่ 6: Tree, Binary Tree และ Binary Search Tree> **รายวิชา:** Data Structures and Algorithms (สำหรับนิสิต IT ชั้นปีที่ 2)> **อ้างอิงแนวเนื้อหา:** Narasimha Karumanchi, *Data Structures and Algorithmic Thinking with Python*> **วิธีใช้:** อ่านคำอธิบาย → รันโค้ดทีละเซลล์ → ลองแก้ค่า input → ทำแบบฝึกหัดท้ายบท (เฉลยอยู่ท้ายไฟล์)---

## 6.1 คำศัพท์พื้นฐาน- **root** = โหนดบนสุด, **leaf** = โหนดที่ไม่มีลูก, **parent / child / sibling**- **depth** ของโหนด = ระยะจาก root, **height** ของต้นไม้ = depth มากสุด- **Binary tree** = แต่ละโหนดมีลูกไม่เกิน 2 (left, right)- **Full** = ทุกโหนดมี 0 หรือ 2 ลูก; **Complete** = เต็มทุกชั้นยกเว้นชั้นล่างสุดที่เติมจากซ้าย- Binary tree ที่มี n โหนด ถ้าสมดุลจะมี height \(\approx \log_2 n\); ถ้าเบ้เป็นเส้นตรงจะเป็น \(n-1\)## 6.2 Traversal| แบบ | ลำดับ | ใช้ทำอะไร ||---|---|---|| Preorder | Root → Left → Right | คัดลอกต้นไม้, serialize || Inorder | Left → Root → Right | ได้ค่าเรียงจากน้อยไปมากใน BST || Postorder | Left → Right → Root | ลบต้นไม้, คำนวณค่าจากล่างขึ้นบน || Level order | ทีละชั้น (BFS) | หาความลึก, พิมพ์เป็นชั้น |

In [ ]:
from collections import dequeclass TreeNode:    __slots__ = ('val', 'left', 'right')    def __init__(self, val, left=None, right=None):        self.val, self.left, self.right = val, left, right# สร้างต้นไม้ตัวอย่าง#         1#       /   \#      2     3#     / \     \#    4   5     6root = TreeNode(1,                TreeNode(2, TreeNode(4), TreeNode(5)),                TreeNode(3, None, TreeNode(6)))def preorder(n, out=None):    out = [] if out is None else out    if n:        out.append(n.val); preorder(n.left, out); preorder(n.right, out)    return outdef inorder(n, out=None):    out = [] if out is None else out    if n:        inorder(n.left, out); out.append(n.val); inorder(n.right, out)    return outdef postorder(n, out=None):    out = [] if out is None else out    if n:        postorder(n.left, out); postorder(n.right, out); out.append(n.val)    return outdef level_order(root):    # BFS ด้วยคิว -> คืนผลแยกเป็นชั้น    if not root: return []    out, q = [], deque([root])    while q:        level = []        for _ in range(len(q)):            n = q.popleft()            level.append(n.val)            if n.left:  q.append(n.left)            if n.right: q.append(n.right)        out.append(level)    return outprint("preorder   :", preorder(root))print("inorder    :", inorder(root))print("postorder  :", postorder(root))print("level order:", level_order(root))

In [ ]:
# Traversal แบบ iterative (ใช้ stack แทน recursion) - เลี่ยง stack overflow ในต้นไม้ลึกมากdef preorder_iter(root):    out, st = [], [root] if root else []    while st:        n = st.pop()        out.append(n.val)        if n.right: st.append(n.right)   # ใส่ right ก่อน เพื่อให้ left ถูก pop ก่อน        if n.left:  st.append(n.left)    return outdef inorder_iter(root):    out, st, cur = [], [], root    while st or cur:        while cur:                        # เดินลงซ้ายให้สุด            st.append(cur); cur = cur.left        cur = st.pop()        out.append(cur.val)        cur = cur.right    return outprint(preorder_iter(root))print(inorder_iter(root))

In [ ]:
# คุณสมบัติพื้นฐานของต้นไม้ (คำนวณแบบ recursive ทั้งหมด O(n))def height(n):    return -1 if n is None else 1 + max(height(n.left), height(n.right))def count_nodes(n):    return 0 if n is None else 1 + count_nodes(n.left) + count_nodes(n.right)def count_leaves(n):    if n is None: return 0    if not n.left and not n.right: return 1    return count_leaves(n.left) + count_leaves(n.right)def is_balanced(n):    # คืน (สมดุลหรือไม่, ความสูง) เพื่อไม่ต้องคำนวณ height ซ้ำ -> O(n)    def dfs(x):        if x is None: return True, -1        lb, lh = dfs(x.left)        rb, rh = dfs(x.right)        return (lb and rb and abs(lh - rh) <= 1), 1 + max(lh, rh)    return dfs(n)[0]def diameter(n):    # เส้นทางที่ยาวที่สุดระหว่างสองโหนดใด ๆ (นับจำนวน edge)    best = 0    def dfs(x):        nonlocal best        if x is None: return -1        lh, rh = dfs(x.left), dfs(x.right)        best = max(best, lh + rh + 2)        return 1 + max(lh, rh)    dfs(n)    return bestprint("height =", height(root), "| nodes =", count_nodes(root),      "| leaves =", count_leaves(root))print("balanced? =", is_balanced(root), "| diameter =", diameter(root))

## 6.3 Binary Search Tree (BST)**คุณสมบัติ:** สำหรับทุกโหนด → ค่าในซับทรีซ้าย < ค่าโหนด < ค่าในซับทรีขวา| Operation | เฉลี่ย | แย่สุด (ต้นไม้เบ้) ||---|---|---|| search | \(O(\log n)\) | \(O(n)\) || insert | \(O(\log n)\) | \(O(n)\) || delete | \(O(\log n)\) | \(O(n)\) |การลบมี 3 กรณี: ไม่มีลูก / มีลูก 1 ตัว / มีลูก 2 ตัว (แทนด้วย **inorder successor**)ปัญหาต้นไม้เบ้แก้ด้วย **self-balancing tree** เช่น AVL, Red-Black Tree

In [ ]:
class BST:    def __init__(self):        self.root = None        self.n = 0    def insert(self, v):                          # O(h)        def go(node):            if node is None:                self.n += 1                return TreeNode(v)            if v < node.val:   node.left  = go(node.left)            elif v > node.val: node.right = go(node.right)            # v == node.val -> ไม่ใส่ซ้ำ            return node        self.root = go(self.root)    def search(self, v):                          # O(h) แบบ iterative        cur = self.root        while cur:            if v == cur.val: return True            cur = cur.left if v < cur.val else cur.right        return False    def min_value(self):        cur = self.root        while cur.left: cur = cur.left             # ซ้ายสุด = น้อยสุด        return cur.val    def max_value(self):        cur = self.root        while cur.right: cur = cur.right        return cur.val    def delete(self, v):                           # O(h)        def go(node, v):            if node is None: return None            if v < node.val:                node.left = go(node.left, v)            elif v > node.val:                node.right = go(node.right, v)            else:                if node.left is None:  return node.right   # กรณี 0-1 ลูก                if node.right is None: return node.left                succ = node.right                          # กรณี 2 ลูก                while succ.left: succ = succ.left           # inorder successor                node.val = succ.val                node.right = go(node.right, succ.val)            return node        self.root = go(self.root, v)    def inorder(self):        return inorder(self.root)                  # ได้ผลเรียงจากน้อยไปมากbst = BST()for v in (50, 30, 70, 20, 40, 60, 80):    bst.insert(v)print("inorder   :", bst.inorder())print("min / max :", bst.min_value(), "/", bst.max_value())print("search 40 :", bst.search(40), "| search 45:", bst.search(45))bst.delete(30)                                     # ลบโหนดที่มี 2 ลูกprint("หลังลบ 30 :", bst.inorder())

In [ ]:
# ตรวจสอบความถูกต้องของ BST และหา LCA (Lowest Common Ancestor)def is_bst(node, lo=float('-inf'), hi=float('inf')):    # ต้องส่งช่วงค่าที่อนุญาตลงไป การเทียบแค่กับลูกตรง ๆ ไม่เพียงพอ    if node is None: return True    if not (lo < node.val < hi): return False    return is_bst(node.left, lo, node.val) and is_bst(node.right, node.val, hi)def lca_bst(node, a, b):    # ใช้คุณสมบัติ BST ตัดสินทิศทาง -> O(h)    while node:        if a < node.val and b < node.val:   node = node.left        elif a > node.val and b > node.val: node = node.right        else:                               return node.valdef kth_smallest(root, k):    # inorder แบบ iterative แล้วหยุดเมื่อครบ k -> O(h + k)    st, cur = [], root    while st or cur:        while cur:            st.append(cur); cur = cur.left        cur = st.pop()        k -= 1        if k == 0: return cur.val        cur = cur.rightprint("is_bst(bst) =", is_bst(bst.root), "| is_bst(root ธรรมดา) =", is_bst(root))print("LCA(20, 40) =", lca_bst(bst.root, 20, 40))print("ค่าน้อยอันดับ 3 =", kth_smallest(bst.root, 3))

---## แบบฝึกหัดบทที่ 61. สร้าง BST ที่สมดุลจากอาร์เรย์ที่เรียงแล้ว (ให้ height น้อยสุด)2. พิมพ์ต้นไม้แบบ zigzag / spiral level order3. ตรวจว่าต้นไม้สองต้น identical กันหรือไม่ และหา mirror image ของต้นไม้4. หา LCA ของ binary tree ทั่วไป (ไม่ใช่ BST)

### เฉลยบทที่ 6

In [ ]:
from collections import deque# ข้อ 1: เลือกตัวกลางเป็น root แบบ recursive -> height = O(log n)def sorted_array_to_bst(a):    if not a: return None    mid = len(a) // 2    return TreeNode(a[mid], sorted_array_to_bst(a[:mid]), sorted_array_to_bst(a[mid+1:]))# ข้อ 2: สลับทิศการเก็บผลในแต่ละชั้นdef zigzag(root):    if not root: return []    out, q, ltr = [], deque([root]), True    while q:        level = deque()        for _ in range(len(q)):            n = q.popleft()            if ltr: level.append(n.val)            else:   level.appendleft(n.val)            if n.left:  q.append(n.left)            if n.right: q.append(n.right)        out.append(list(level)); ltr = not ltr    return out# ข้อ 3def identical(p, q):    if p is None and q is None: return True    if p is None or q is None:  return False    return p.val == q.val and identical(p.left, q.left) and identical(p.right, q.right)def mirror(n):    if n is None: return None    return TreeNode(n.val, mirror(n.right), mirror(n.left))   # สลับซ้าย-ขวา# ข้อ 4: LCA ทั่วไป - ถ้าเจอ a หรือ b ให้คืนตัวเอง; โหนดที่ได้คำตอบจากทั้งสองข้างคือ LCAdef lca_general(node, a, b):    if node is None or node.val in (a, b):        return node    l = lca_general(node.left, a, b)    r = lca_general(node.right, a, b)    if l and r: return node    return l or rbal = sorted_array_to_bst([1,2,3,4,5,6,7])print("balanced inorder:", inorder(bal), "| height =", height(bal))print("zigzag:", zigzag(root))print("identical:", identical(root, root), "| mirror preorder:", preorder(mirror(root)))print("LCA(4,5) =", lca_general(root, 4, 5).val)